# Notebook 03 — Análisis de Anomalías en Formularios E14

Aplica reglas matemáticas y z-score sobre los valores extraídos por OCR para detectar inconsistencias.

**Input:** `../data/output/ocr_results.csv` — generado por notebook 02  
**Formato esperado:** columnas `form_id`, `field`, `ocr_value`  
**Output:** `../data/output/resultados_anomalias.csv`

In [ ]:
import pandas as pd
import numpy as np

OCR_CSV  = '../data/output/ocr_results.csv'
OUT_CSV  = '../data/output/resultados_anomalias.csv'
UMBRAL_Z = 2.5

df_ocr = pd.read_csv(OCR_CSV)
print(f'Registros OCR cargados: {len(df_ocr)}')
print(f'Formularios únicos: {df_ocr["form_id"].nunique()}')
df_ocr.head(10)

In [ ]:
# Pivot: una fila por formulario, columnas = campos del E14
df = df_ocr.pivot_table(index='form_id', columns='field', values='ocr_value', aggfunc='first')
df.columns.name = None
df = df.reset_index()

# Convertir a numérico (falla OCR → NaN)
campos_num = [
    'total_sufragantes', 'votos_en_urna', 'votos_incinerados',
    'votos_candidato_1', 'votos_candidato_2',
    'votos_blanco', 'votos_nulos', 'votos_no_marcados', 'total_mesa'
]
for col in campos_num:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Formularios en tabla: {len(df)}')
print('Campos con NaN (OCR fallido):')
print(df[[c for c in campos_num if c in df.columns]].isna().sum())
df.head()

## Reglas matemáticas E14

| Regla | Condición | Qué detecta |
|-------|-----------|-------------|
| `alerta_total` | `total_mesa ≠ candidato_1 + candidato_2 + blanco + nulos + no_marcados` | Total alterado |
| `alerta_sufragantes` | `total_sufragantes ≠ votos_en_urna + votos_incinerados` | Sufragantes no cuadran |
| `alerta_ocr_invalido` | Campo clave con NaN | OCR no pudo leer el valor |

In [ ]:
# Regla 1: total_mesa == suma de votos detallados
cols_detalle = ['votos_candidato_1', 'votos_candidato_2',
                'votos_blanco', 'votos_nulos', 'votos_no_marcados']

df['suma_detalle']  = df[[c for c in cols_detalle if c in df.columns]].sum(axis=1, min_count=1)
df['diff_total']    = (df['total_mesa'] - df['suma_detalle']).abs()
df['alerta_total']  = df['diff_total'] > 0

# Regla 2: total_sufragantes == votos_en_urna + votos_incinerados
df['suma_urna_inc']      = df['votos_en_urna'] + df['votos_incinerados']
df['diff_sufragantes']   = (df['total_sufragantes'] - df['suma_urna_inc']).abs()
df['alerta_sufragantes'] = df['diff_sufragantes'] > 0

# Regla 3: campos clave ilegibles
campos_clave = [c for c in ['total_mesa', 'total_sufragantes', 'votos_en_urna'] if c in df.columns]
df['alerta_ocr_invalido'] = df[campos_clave].isna().any(axis=1)

df['sospechosa_reglas'] = (
    df['alerta_total'] |
    df['alerta_sufragantes'] |
    df['alerta_ocr_invalido']
)

print('=== Resultados reglas matemáticas ===')
print(f"alerta_total         : {df['alerta_total'].sum()} formularios")
print(f"alerta_sufragantes   : {df['alerta_sufragantes'].sum()} formularios")
print(f"alerta_ocr_invalido  : {df['alerta_ocr_invalido'].sum()} formularios")
print(f"sospechosa_reglas    : {df['sospechosa_reglas'].sum()} formularios")

## Z-score estadístico

Compara cada formulario contra el conjunto.  
Si `|z| > 2.5` en cualquier campo → formulario atípico.

In [ ]:
cols_zscore = [c for c in campos_num if c in df.columns]

for col in cols_zscore:
    media = df[col].mean()
    std   = df[col].std()
    df[f'z_{col}'] = (df[col] - media) / std if std > 0 else 0.0

zcols = [f'z_{c}' for c in cols_zscore]
df['max_zscore']        = df[zcols].abs().max(axis=1)
df['sospechosa_zscore'] = df['max_zscore'] > UMBRAL_Z

print(f'=== Top formularios por z-score (umbral={UMBRAL_Z}) ===')
top = df.nlargest(10, 'max_zscore')[['form_id', 'max_zscore', 'sospechosa_zscore']]
print(top.to_string(index=False))

In [ ]:
df['sospechosa_final'] = df['sospechosa_reglas'] | df['sospechosa_zscore']

def motivo(row):
    m = []
    if row['alerta_total']:        m.append('total_incorrecto')
    if row['alerta_sufragantes']:  m.append('sufragantes_no_cuadran')
    if row['alerta_ocr_invalido']: m.append('ocr_invalido')
    if row['sospechosa_zscore']:   m.append(f'zscore={row["max_zscore"]:.2f}')
    return ' | '.join(m) if m else ''

df['motivo_alerta'] = df.apply(motivo, axis=1)

print('=== Resumen final ===')
cols_resumen = ['form_id', 'sospechosa_reglas', 'sospechosa_zscore', 'sospechosa_final', 'motivo_alerta']
print(df[cols_resumen].to_string(index=False))
print(f"\nTotal sospechosas: {df['sospechosa_final'].sum()} / {len(df)}")

In [ ]:
cols_export = (
    ['form_id'] +
    [c for c in campos_num if c in df.columns] +
    ['suma_detalle', 'diff_total', 'diff_sufragantes',
     'alerta_total', 'alerta_sufragantes', 'alerta_ocr_invalido',
     'sospechosa_reglas', 'max_zscore', 'sospechosa_zscore',
     'sospechosa_final', 'motivo_alerta']
)
cols_export = [c for c in cols_export if c in df.columns]

df[cols_export].to_csv(OUT_CSV, index=False)
print(f'Exportado: {OUT_CSV}')
print(f'Filas: {len(df)} | Columnas: {len(cols_export)}')